<a href="https://colab.research.google.com/github/Mihailby/Taxes/blob/main/notebooks/01/tut_1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.2 The two-variable linear program in AMPL

In [132]:
%pip install PyPDF2 reportlab pandas
import PyPDF2
from PyPDF2 import PdfReader, PdfWriter
import pandas as pd

In [133]:
# @markdown
Name = 'Mikhail Riabtsev' # @param {type:"string"}
SSN = '123-45-6789' # @param {type:"string"}
import re
def split_ssn(ssn):
    # Check if SSN is in correct format
    ssn_pattern = r'^\d{3}-\d{2}-\d{4}$'

    if not re.match(ssn_pattern, ssn):
        print("❌ Invalid SSN format. Please use XXX-XX-XXXX")
        return None, None, None

    # Split the SSN
    parts = ssn.split('-')
    SSN1 = parts[0]  # First 3 digits
    SSN2 = parts[1]  # Middle 2 digits
    SSN3 = parts[2]  # Last 4 digits

    return SSN1, SSN2, SSN3

# Split the SSN
ssn1, ssn2, ssn3 = split_ssn(SSN)

# @markdown
Street_Address = "dd"  # @param {type:"string"}
City_Town_County_ZIP_Code = "s"  # @param {type:"string"}

# Default placeholder texts
DEFAULT_ADDRESS = "Street address, apartment number, or rural route number"
DEFAULT_CITY = "City or town, state or province, and country. Include ZIP code or postal code where appropriate."

# Check if values are set (not the default placeholders)
address_filled = Street_Address and Street_Address != DEFAULT_ADDRESS
city_filled = City_Town_County_ZIP_Code and City_Town_County_ZIP_Code != DEFAULT_CITY

if address_filled and city_filled:
    print("\n✅ All address fields are filled!")
else:
    print("\n⚠️ Please fill in the following:")
    if not address_filled:
        print("  • Street address")
    if not city_filled:
        print("  • City/Town/County/ZIP")


✅ All address fields are filled!


In [136]:
!curl -L -o Schedule_C.pdf \
https://raw.githubusercontent.com/Mihailby/Taxes/main/Schedule_C.pdf

def fill_pdf_form_fields(input_path, output_path, values):
    reader = PdfReader(input_path)
    writer = PdfWriter()

    for page in reader.pages:
        writer.add_page(page)

    if reader.get_fields():
        writer.update_page_form_field_values(writer.pages[0], values)
    else:
        print("⚠️ No form fields detected in PDF")

    with open(output_path, "wb") as f:
        writer.write(f)

    print(f"PDF saved to: {output_path}")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  145k  100  145k    0     0   653k      0 --:--:-- --:--:-- --:--:--  656k


In [137]:
field_values = {
    "Name": Name,
    "SSN 1": ssn1,
    "SSN 2": ssn2,
    "SSN 3": ssn3,
    "Address Where Rented": "123 Main St, NY"
}

fill_pdf_form_fields(
    "Schedule_C.pdf",
    "filled_Schedule_C.pdf",
    field_values
)

from google.colab import files
files.download("filled_Schedule_C.pdf")

PDF saved to: filled_Schedule_C.pdf


## Add Links

In [139]:
# Install PyMuPDF library for PDF manipulation
%pip install -q pymupdf
import fitz, os
from google.colab import files

# Download the source PDF from GitHub and save as S.pdf
!wget -q -O S.pdf https://raw.githubusercontent.com/Mihailby/Taxes/main/Schedule_C.pdf

def add_circles(in_pdf, out_pdf, text, url, size=5, fill=(1,0.95,0.85), stroke=(0.8,0.5,0.1)):
    doc = fitz.open(in_pdf)   # Open the input PDF
    total = 0                 # Counter for total circles added

    # Loop through each page in the PDF
    for p, page in enumerate(doc):
        for r in page.search_for(text):         # Find all instances of the target text on this page
            # Calculate position: to the right of text (x) and vertically centered (y)
            x = r.x1 + 1      # 1 point to the right of text
            y = r.y0 + (r.y1 - r.y0 - size) / 2 # Center circle vertically

            rect = fitz.Rect(x, y, x + size, y + size)  # Create rectangle that defines the circle's bounds
            page.insert_link({"kind": fitz.LINK_URI, "from": rect, "uri": url}) # Make the rectangle clickable - opens URL when clicked
            a = page.add_circle_annot(rect)     # Create a circle annotation (visible on screen)
            a.set_colors(stroke=stroke, fill=fill)  # Set circle colors: border (stroke) and interior (fill)
            a.set_border(width=0.5)             # Set border thickness

            # Make circle NOT printable (screen only)
            a.set_flags(a.flags & ~fitz.PDF_ANNOT_IS_PRINT) # Remove the print flag from annotation flags
            a.update()                          # Apply changes to annotation

            total += 1                          # Increment counter

        if page.search_for(text):               # Print progress for this page if circles were added
            print(f"✅ Page {p+1}: added {len(page.search_for(text))} circle(s)")

    doc.save(out_pdf)                           # Save the modified PDF
    doc.close()                                 # Close the document

    # Print final summary
    #print(f"\n✅ Done! Added {total} circles → {out_pdf}")

# Add clickable circles after "Schedule C: Deductions" text
add_circles("S.pdf", "out.pdf", "Schedule C: Deductions", "https://ampl.com")

files.download("out.pdf")                       # Download the resulting PDF to local machine

✅ Page 1: added 1 circle(s)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>